# preflight_e17
Phase 2 Stage 1 E17 DEV-C preflight and authorization (Colab CPU). Run cells 2, 3, and 4 in order. Configure only cell 3. Use the pinned CPU runtime and exact Phase 2 Stage 1 commit.

- Set `MODEL = 'both'` (default) to verify and authorize all 8 DEV-C midpoint feature releases, 8 frozen Stage 3 ordinary heads + source-FIT scalers, both `en -> en` DEV-C diagonal `atol=0, rtol=0` self-checks, and all 24 off-diagonal DEV-C routes in one immutable `phase2_e17_auth` artifact under `PHASE1_ROOT/phase2/plans/`.
- Set `MODEL = 'qwen'` or `MODEL = 'gemma'` to preflight a single model.
- Phase 1 directories remain strictly read-only; FIT feature matrices and `CONFIRM` are never opened.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Research-phase2-e17-preflight')
CODE_REF = 'phase2'  # Set to the approved Phase 2 Stage 1 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 1 settings (edit only this cell).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
MODEL = 'both'          # 'both' is required for Stage 1 verify-stage1-bundle ('qwen' or 'gemma' for single-model checks).
STAGE4_RELEASE = None   # None uses the pinned accepted Phase 1 Stage 4 release; or Path / {'path': ..., 'sha256': ...}.
RESUME_REF = None       # Optional explicit {'path': ..., 'sha256': ...} or Path to an existing phase2_e17_auth _COMPLETE.json.
OUTPUT_REF = None       # Optional mounted JSON path (e.g. PHASE1_ROOT / 'phase2/plans/e17_auth_ref.json').


In [ ]:
import json, signal
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if MODEL not in ('qwen', 'gemma', 'both'):
    raise RuntimeError("Set MODEL to 'qwen', 'gemma', or 'both'.")
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')
command = [
    sys.executable, '-u', '-m', 'src.phase2.e17_auth',
    '--phase1-root', str(PHASE1_ROOT),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if STAGE4_RELEASE is not None:
    if isinstance(STAGE4_RELEASE, dict):
        rel_path = PHASE1_ROOT / STAGE4_RELEASE['path']
        if not rel_path.is_file():
            raise RuntimeError(f'Missing Stage 4 release manifest: {rel_path}')
        command += ['--stage4-release', json.dumps(STAGE4_RELEASE)]
    else:
        rel_path = Path(STAGE4_RELEASE)
        if not rel_path.is_file():
            raise RuntimeError(f'Missing Stage 4 release manifest: {rel_path}')
        command += ['--stage4-release', str(rel_path)]
if RESUME_REF is not None:
    if isinstance(RESUME_REF, dict):
        res_path = PHASE1_ROOT / RESUME_REF['path']
        if not res_path.is_file():
            raise RuntimeError(f'Missing explicit resume manifest: {res_path}')
        command += ['--resume-ref', json.dumps(RESUME_REF)]
    else:
        res_path = Path(RESUME_REF)
        if not res_path.is_file():
            raise RuntimeError(f'Missing explicit resume manifest: {res_path}')
        command += ['--resume-ref', str(res_path)]
if OUTPUT_REF is not None:
    command += ['--output-ref', str(OUTPUT_REF)]
process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
try:
    for line in process.stdout:
        print(line, end='', flush=True)
        try:
            payload = json.loads(line)
            if isinstance(payload, dict) and payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
        except json.JSONDecodeError:
            pass
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
code = process.wait()
if code != 0:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E17 DEV-C preflight blocked/failed: {detail}')
